# 02 - U-Net Baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfefGuesmi/msdd-arab-youth-space-hackathon/blob/main/MSDD-Net/notebooks/02_UNet_Baseline.ipynb)

Trains the MARIDA paper U-Net baseline (narrow bottleneck, weighted cross-entropy).

Runtime: about 45 to 90 minutes on a Colab T4 GPU.

## 1. Setup

In [ ]:
import os
if not os.path.isdir('msdd-arab-youth-space-hackathon'):
    !git clone -b msddnet https://github.com/AfefGuesmi/msdd-arab-youth-space-hackathon.git

%cd msdd-arab-youth-space-hackathon/MSDD-Net

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Verify data

In [ ]:
!ls data/patches | head -3
!wc -l data/splits/train_X.txt data/splits/val_X.txt data/splits/test_X.txt

## 3. Train U-Net

Paper recipe: Adam, lr 2e-4, x0.1 at epoch 40, batch 5, 45 epochs, weighted CE with c=1.03.

In [ ]:
!python train_marida.py \
  --model unet \
  --data_path data \
  --epochs 45 \
  --batch 5 \
  --lr 2e-4 \
  --scheduler multistep \
  --lr_steps '[40]' \
  --weight_param 1.03

## 4. Plot the training curve

In [ ]:
import json
import matplotlib.pyplot as plt

with open('trained_models/unet/history.json') as f:
    h = json.load(f)

ep   = [x['epoch'] for x in h]
tr   = [x['train_loss'] for x in h]
va   = [x['val_loss'] for x in h]
miou = [x['val_miou'] for x in h]
mf1  = [x['val_mf1'] for x in h]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(ep, tr, label='train')
axes[0].plot(ep, va, label='val')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].set_title('U-Net loss')
axes[1].plot(ep, miou, label='val mIoU')
axes[1].plot(ep, mf1,  label='val mF1')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('score'); axes[1].legend(); axes[1].set_title('U-Net val scores')
plt.tight_layout(); plt.show()

## 5. Evaluate on the MARIDA test set

In [ ]:
!python evaluation_swin_unetv2.py \
  --model_type unet \
  --backbone none \
  --hidden_channels 16 \
  --model_path trained_models/unet/best_model.pth \
  --tta False

## 6. Show per-class results

In [ ]:
import json, os, glob
p = 'logs/evaluating_unet_unet_results.json'
if os.path.isfile(p):
    with open(p) as f:
        r = json.load(f)
    import pandas as pd
    rows = []
    for cls, m in r['per_class'].items():
        rows.append({'class': cls, **m})
    display(pd.DataFrame(rows))
else:
    print('No results file yet; run the previous cell first.')